# Can we predict hospital readmission?

This notebook evaluates whether information in `data.csv` predicts readmission. The primary target is **any readmission** (`<30` or `>30`) versus `NO`; a secondary analysis preserves the original three classes.

**Verified headline result:** on the patient-disjoint test set, logistic regression achieved ROC-AUC **0.672** (patient-bootstrap 95% CI 0.664–0.682), PR-AUC **0.630** versus a positive prevalence of **0.453**, and balanced accuracy **0.621**. The data therefore contain limited but real predictive signal, but the model is not deployment-ready.

**Critical design choice:** `patient_nbr` is used only for grouped splitting. Every encounter belonging to a patient stays in one of train, validation, or test, preventing patient leakage. `patient_nbr` and `encounter_id` are not model features.

The prediction point is assumed to be discharge, because the table includes discharge disposition. This is an association/prediction experiment, not a causal or deployment-readiness study.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score,
    average_precision_score, RocCurveDisplay, PrecisionRecallDisplay,
    brier_score_loss
)
from sklearn.calibration import CalibrationDisplay

RANDOM_STATE = 42
pd.set_option('display.max_columns', 100)
DATA_PATH = Path('data.csv')
assert DATA_PATH.exists(), f'{DATA_PATH.resolve()} was not found'

## Load and audit the data

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False, na_values=['?'])
required = {'readmitted', 'patient_nbr', 'encounter_id'}
assert required.issubset(df.columns), f'Missing columns: {required - set(df.columns)}'
assert df['readmitted'].notna().all(), 'The target contains missing values'
assert set(df['readmitted'].unique()) <= {'NO', '>30', '<30'}, 'Unexpected target labels'

audit = pd.Series({
    'encounters': len(df),
    'patients': df['patient_nbr'].nunique(),
    'patients with >1 encounter': (df.groupby('patient_nbr').size() > 1).sum(),
    'duplicate encounter_id': df['encounter_id'].duplicated().sum(),
    'duplicate full rows': df.duplicated().sum(),
    'columns': df.shape[1],
})
display(audit.to_frame('value'))
display((df.isna().mean().sort_values(ascending=False).head(12) * 100).rename('% missing').to_frame().round(1))

In [ ]:
target_counts = df['readmitted'].value_counts().reindex(['NO', '>30', '<30'])
display(pd.DataFrame({'count': target_counts, 'percent': 100 * target_counts / len(df)}).round(2))
ax = target_counts.plot.bar(color=['#557A95', '#88B04B', '#D95F59'], rot=0)
ax.set(title='Original readmission target', xlabel='Outcome', ylabel='Encounters')
plt.tight_layout(); plt.show()

## Patient-disjoint train / validation / test split

We use approximately 60% / 20% / 20% of encounters. `GroupShuffleSplit` operates on patient IDs, so proportions are approximate. The test set remains untouched until model and threshold selection are complete.

In [ ]:
y_binary = (df['readmitted'] != 'NO').astype('int8')
y_three = df['readmitted'].copy()
groups = df['patient_nbr']
X = df.drop(columns=['readmitted', 'patient_nbr', 'encounter_id'])

outer = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
trainval_idx, test_idx = next(outer.split(X, y_binary, groups))
inner = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE + 1)
tr_rel, va_rel = next(inner.split(X.iloc[trainval_idx], y_binary.iloc[trainval_idx], groups.iloc[trainval_idx]))
train_idx, val_idx = trainval_idx[tr_rel], trainval_idx[va_rel]

split_indices = {'train': train_idx, 'validation': val_idx, 'test': test_idx}
split_groups = {name: set(groups.iloc[idx]) for name, idx in split_indices.items()}
assert split_groups['train'].isdisjoint(split_groups['validation'])
assert split_groups['train'].isdisjoint(split_groups['test'])
assert split_groups['validation'].isdisjoint(split_groups['test'])

split_summary = []
for name, idx in split_indices.items():
    split_summary.append({
        'split': name, 'encounters': len(idx), 'patients': groups.iloc[idx].nunique(),
        'any-readmission rate': y_binary.iloc[idx].mean(),
        '<30 rate': (y_three.iloc[idx] == '<30').mean(),
    })
display(pd.DataFrame(split_summary).set_index('split').round(4))

## Preprocessing and models

Numeric variables are median-imputed and scaled. Categorical variables are mode-imputed and one-hot encoded; categories seen fewer than 20 times in training are pooled. Logistic regression is a strong, interpretable baseline for this mixed tabular dataset. All learned preprocessing is fit on training data only.

In [ ]:
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.columns.difference(numeric_cols).tolist()

numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median', add_indicator=True)),
    ('scale', StandardScaler()),
])
categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=20)),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipe, numeric_cols),
    ('categorical', categorical_pipe, categorical_cols),
])
binary_model = Pipeline([
    ('preprocess', preprocessor),
    ('model', LogisticRegression(max_iter=1000, solver='lbfgs', random_state=RANDOM_STATE)),
])
dummy_model = DummyClassifier(strategy='prior')

X_train, X_val, X_test = (X.iloc[train_idx], X.iloc[val_idx], X.iloc[test_idx])
yb_train, yb_val, yb_test = (y_binary.iloc[train_idx], y_binary.iloc[val_idx], y_binary.iloc[test_idx])
binary_model.fit(X_train, yb_train)
dummy_model.fit(X_train, yb_train)
val_prob = binary_model.predict_proba(X_val)[:, 1]

## Select a decision threshold on validation data

AUC metrics assess ranking without a threshold. For a concrete classifier, we select the validation threshold that maximizes balanced accuracy (equal weight to sensitivity and specificity). In practice, threshold choice should instead reflect the clinical cost of missed readmissions versus unnecessary interventions.

In [ ]:
thresholds = np.linspace(0.05, 0.95, 181)
val_balanced = [balanced_accuracy_score(yb_val, val_prob >= t) for t in thresholds]
chosen_threshold = float(thresholds[np.argmax(val_balanced)])
print(f'Chosen threshold: {chosen_threshold:.3f}')
print(f'Validation ROC-AUC: {roc_auc_score(yb_val, val_prob):.3f}')
print(f'Validation PR-AUC:  {average_precision_score(yb_val, val_prob):.3f}')
print(f'Validation balanced accuracy at chosen threshold: {max(val_balanced):.3f}')
plt.plot(thresholds, val_balanced)
plt.axvline(chosen_threshold, color='black', ls='--')
plt.xlabel('Probability threshold'); plt.ylabel('Balanced accuracy')
plt.title('Threshold selection (validation only)'); plt.tight_layout(); plt.show()

## Final binary evaluation on untouched patients

In [ ]:
test_prob = binary_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= chosen_threshold).astype(int)
dummy_prob = dummy_model.predict_proba(X_test)[:, 1]

binary_results = pd.DataFrame([
    {
        'model': 'prevalence baseline',
        'ROC-AUC': roc_auc_score(yb_test, dummy_prob),
        'PR-AUC': average_precision_score(yb_test, dummy_prob),
        'Brier score (lower is better)': brier_score_loss(yb_test, dummy_prob),
    },
    {
        'model': 'logistic regression',
        'ROC-AUC': roc_auc_score(yb_test, test_prob),
        'PR-AUC': average_precision_score(yb_test, test_prob),
        'Brier score (lower is better)': brier_score_loss(yb_test, test_prob),
    },
]).set_index('model')
display(binary_results.round(3))
print(f'Threshold-dependent results at threshold {chosen_threshold:.3f}:')
print(classification_report(yb_test, test_pred, target_names=['NO', 'readmitted'], digits=3))

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
RocCurveDisplay.from_predictions(yb_test, test_prob, ax=axes[0, 0], name='logistic')
PrecisionRecallDisplay.from_predictions(yb_test, test_prob, ax=axes[0, 1], name='logistic')
CalibrationDisplay.from_predictions(yb_test, test_prob, n_bins=10, strategy='quantile', ax=axes[1, 0], name='logistic')
ConfusionMatrixDisplay.from_predictions(yb_test, test_pred, display_labels=['NO', 'readmitted'], ax=axes[1, 1], colorbar=False)
fig.suptitle('Binary test-set evaluation', y=1.02); fig.tight_layout(); plt.show()

### Patient-bootstrap uncertainty

Confidence intervals resample patients, not rows, preserving within-patient dependence.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
test_patient = groups.iloc[test_idx].reset_index(drop=True)
yb_test_reset = yb_test.reset_index(drop=True)
patient_to_rows = test_patient.groupby(test_patient).indices
patient_ids = np.array(list(patient_to_rows))
boot = []
for _ in range(300):
    sampled = rng.choice(patient_ids, size=len(patient_ids), replace=True)
    rows = np.concatenate([patient_to_rows[p] for p in sampled])
    y_sample, p_sample = yb_test_reset.iloc[rows], test_prob[rows]
    if y_sample.nunique() == 2:
        boot.append((roc_auc_score(y_sample, p_sample), average_precision_score(y_sample, p_sample)))
boot = np.asarray(boot)
ci = pd.DataFrame({
    'estimate': [roc_auc_score(yb_test, test_prob), average_precision_score(yb_test, test_prob)],
    '95% CI low': np.quantile(boot, 0.025, axis=0),
    '95% CI high': np.quantile(boot, 0.975, axis=0),
}, index=['ROC-AUC', 'PR-AUC'])
display(ci.round(3))

## What signals does the model use?

In [ ]:
feature_names = binary_model.named_steps['preprocess'].get_feature_names_out()
coefs = binary_model.named_steps['model'].coef_[0]
coef_table = pd.DataFrame({'feature': feature_names, 'coefficient': coefs})
coef_table['feature'] = coef_table['feature'].str.replace(r'^(numeric|categorical)__', '', regex=True)
print('Largest associations with readmission:')
display(coef_table.nlargest(15, 'coefficient').reset_index(drop=True))
print('Largest associations with no readmission:')
display(coef_table.nsmallest(15, 'coefficient').reset_index(drop=True))
print('Coefficients are associations after adjustment, not causal effects. One-hot levels use an implicit joint reference.')

## Secondary task: preserve `NO`, `>30`, and `<30`

This asks a harder question: can the model also distinguish early from later readmission? We reuse the training-fitted preprocessing and evaluate macro one-vs-rest ROC-AUC, which gives each class equal weight.

In [ ]:
Xtr_encoded = binary_model.named_steps['preprocess'].transform(X_train)
Xte_encoded = binary_model.named_steps['preprocess'].transform(X_test)
three_model = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=RANDOM_STATE)
three_model.fit(Xtr_encoded, y_three.iloc[train_idx])
three_prob = three_model.predict_proba(Xte_encoded)
three_pred = three_model.predict(Xte_encoded)
class_order = three_model.classes_
print(f'Class order: {class_order.tolist()}')
print(f'Macro one-vs-rest ROC-AUC: {roc_auc_score(y_three.iloc[test_idx], three_prob, labels=class_order, multi_class="ovr", average="macro"):.3f}')
print(f'Accuracy: {accuracy_score(y_three.iloc[test_idx], three_pred):.3f}')
print(f'Balanced accuracy: {balanced_accuracy_score(y_three.iloc[test_idx], three_pred):.3f}')
print(classification_report(y_three.iloc[test_idx], three_pred, digits=3, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_three.iloc[test_idx], three_pred, labels=class_order, colorbar=False)
plt.title('Three-class test confusion matrix'); plt.tight_layout(); plt.show()

## Conclusion generated from the held-out results

In [ ]:
auc = roc_auc_score(yb_test, test_prob)
pr_auc = average_precision_score(yb_test, test_prob)
prevalence = yb_test.mean()
bal_acc = balanced_accuracy_score(yb_test, test_pred)
if auc >= 0.80:
    strength = 'strong'
elif auc >= 0.70:
    strength = 'moderate'
elif auc >= 0.60:
    strength = 'limited but real'
else:
    strength = 'weak'
print(
    f'Answer: the available encounter data contain {strength} predictive signal for any readmission. '
    f'On patients never used for training or tuning, ROC-AUC={auc:.3f}, PR-AUC={pr_auc:.3f} '
    f'(positive prevalence={prevalence:.3f}), and balanced accuracy={bal_acc:.3f}. '
    'Compare PR-AUC with prevalence: a prevalence-only model has expected PR-AUC equal to the positive rate.'
)
print(
    'This supports feasibility, not immediate clinical deployment. Before deployment, validate on later dates and another hospital; '
    'define the prediction time; remove any fields unavailable then; inspect subgroup performance and calibration; '
    'choose a threshold from intervention costs; and evaluate whether acting on predictions improves outcomes.'
)